# Slabs with different vacuum sizes

This jupyter nb serves to do the following:

(1) Exctract the **optimized** coordinates of the 1x2 CaB6 slab configuration, which was initially constructed using 16.52 Å of vacuum but ended with 17.42 Å after vc-relax. This initial slab has to be centered within the simulation cell.

(2) Modify the cell parameters and atomic positions so that we're able to change the vacuum of the slab model.

(3) Export the edited coordinates and parameters as a Quantum ESPRESSO input file. 

In [1]:
import ase
from ase.build import bulk
from ase.io import read,write,cif, espresso
from ase.io.espresso import write_espresso_in
from ase import Atoms
from ase.visualize import view

In [2]:
# Converting the slab from .CIF file into an 'atoms' object with ase
cif_file = 'CaB6_1x2_optimized_slab.cif'
slab = ase.io.cif.read_cif(cif_file)
view(slab, viewer="x3d")

In [3]:
# Retrieving useful data

# Get cell parameters
cell_parameters = slab.get_cell()

# Get cell parameter in z-direction (the height of the entire simulation cell)
z_dimension = cell_parameters[2][2]

# Get the coordinates of all atoms
original_positions = slab.get_positions()

In [4]:
# Creating a slab model where the above slab is set at the bottom of the simulation cell #

bottom_slab = slab.copy()
bottom_slab_positions = bottom_slab.get_positions()

# Recover the z-axis coordinates of all atoms to get the lowest value
z_coordinates_list = []
for position in bottom_slab_positions:
    coordinate_in_z = position[2]
    z_coordinates_list.append(coordinate_in_z)

min_z_value = min(z_coordinates_list) 

# Setting the slab at the bottom of the simulation cell by shifting all atoms towards the bottom accordingly
for position in bottom_slab_positions:
    displacement = position[2] - min_z_value 
    position[2] = displacement

# Accomodating the slab at the bottom by setting the modified coordinates 
bottom_slab.set_positions(bottom_slab_positions)

# Calculating vacuum of the input slab model

# Recover the z-axis coordinates of all atoms in the bottom-accomodated slab to get the highest value
z_coordinates_list_2 = []
for position in bottom_slab_positions:
    coordinate_in_z = position[2]
    z_coordinates_list_2.append(coordinate_in_z)

max_z_value = max(z_coordinates_list_2)

init_vac = z_dimension - max_z_value
print('The vacuum in the slab model:', cif_file, 'is: ', init_vac, 'Å')
view(bottom_slab, viewer="x3d")

The vacuum in the slab model: CaB6_1x2_optimized_slab.cif is:  17.420090191040003 Å


So far we have accomodated the slab model in a more friendly-editable form by setting the beggining of the slab to the bottom of the simulation cell. Now we can tune the vacuum as follows: 

In [5]:
new_vac = 15.0 # user enters the new vacuum of the slab

# Creating the new slab with a different vacuum (only changing the z-dimension)
new_slab = bottom_slab.copy()
new_z_dimension = new_vac + max_z_value
new_slab.set_cell([cell_parameters[0], cell_parameters[1], [0,0,new_z_dimension]])

# Checking if the z-dimension was succesfully changed and if it produces the desired vacuum in the new slab model

# Get cell parameters of the new slab
new_cell_parameters = new_slab.get_cell()

# Get cell parameter in z-direction (the height of the entire simulation cell)
z_dimension_new = new_cell_parameters[2][2]

# Get the coordinates of all atoms in the new slab model
new_slab_positions = new_slab.get_positions()

# Recover the z-axis coordinates of all atoms in the new slab to get the highest value
z_coordinates_list_3 = []
for position in new_slab_positions:
    coordinate_in_z = position[2]
    z_coordinates_list_3.append(coordinate_in_z)

max_z_value_new = max(z_coordinates_list_3)
final_vac = z_dimension_new - max_z_value_new
print('The vacuum in the new slab is: ', final_vac, 'Å')

The vacuum in the new slab is:  15.0 Å


Finally, we have to center the slab

In [6]:
# Calculating some necessary variables

half_max_z_value = max_z_value_new/2 # Divide the highest value in z-direction in the new slab by 2
half_z_dimension = z_dimension_new/2 # Divide the height of the cell of the new slab model by 2
D = half_max_z_value + half_z_dimension

In [7]:
# Copy the array with the coordinates of all atoms in the new slab model (uncentered) to a new array to further modify it
centered_slab_positions = new_slab_positions.copy()

# Creating new (centered) coordinates displacing all atoms by 'D' along the z-direction
for position in centered_slab_positions:
    displacement = D - position[2]
    position[2] = displacement

In [8]:
# Centering the slab by setting the modified coordinates
new_slab.set_positions(centered_slab_positions)
view(new_slab, viewer="x3d")

In [10]:
## GENERATING QE INPUT FILE (TEMPLATE) WITH THE NEW COORDINATES OF THE CENTERED SLAB

# Fake (or real if desired) Pseudopotentials
pseudopotentials={'B': 'B_ONCV_PBE_fr.upf', 'Ca': 'Ca_ONCV_PBE_fr.upf'}

# input data
input_data = {
    'calculation': 'scf',
    'restart_mode': 'from_scratch',
    'ecutwfc': 80,
    'occupations': 'smearing',
    'degauss': 0.005,
    'smearing': 'marzari-vanderbilt',
    'conv_thr': 1e-8,
    'mixing_mode': 'local-TF',
}  # This flat dictionary will be converted to a nested dictionary where, for example, "calculation" will be put into the "control" section

#pw_input_file = './15.0_vac/pw_scf.in'
#write(pw_input_file, new_slab, input_data=input_data, pseudopotentials=pseudopotentials, format='espresso-in')